<a href="https://colab.research.google.com/github/Shwetabh1013/flyrank-ml-internship-shwetabh/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shwetabh1013/flyrank-ml-internship-shwetabh/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Connect to the warehouse

*Same pattern as `notebooks/03_working_with_the_full_release.ipynb` — never paste the token into a cell, this repo is public. In Colab, store it as a Secret named `HF_TOKEN` instead and this cell will pick it up from the environment automatically.*

In [1]:
%pip -q install duckdb huggingface_hub

import os, getpass
import duckdb
import pandas as pd

HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients":       f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content":       f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily":        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "fact_query_90d":    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

# my mid-panel iteration month -- NOT the sealed _sample table (that's June 2026, final month)
MONTH = "2026-03"
REL_MONTH = f"read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/**/*.parquet')"

for name, src in TABLES.items():
    n = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
    print(f"{name:22} {n:>12,} rows")


Paste your Hugging Face READ token (hf_...): ··········
dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


**Schema check first.** I don't want to guess column names -- `DESCRIBE` on a zero-row query shows the real schema without pulling any data.

In [2]:
print("--- fact_content_daily_performance columns ---")
print(con.sql(f"DESCRIBE SELECT * FROM {REL_MONTH} LIMIT 0").df())
print()
print("--- dim_clients columns ---")
print(con.sql(f"DESCRIBE SELECT * FROM {TABLES['dim_clients']} LIMIT 0").df())
print()
print("--- dim_content columns ---")
print(con.sql(f"DESCRIBE SELECT * FROM {TABLES['dim_content']} LIMIT 0").df())


--- fact_content_daily_performance columns ---
                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users 

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**1. What one row means for my lane.** The raw table's grain is one row per `(report_date, client_hash_id, content_hash_id)` -- one content item's search performance on one calendar day for one client. For Lane 2's decision (which pages to review for refresh), the unit that actually matters is coarser: **one content item, over one review month** -- I aggregate the daily rows up to a per-content-per-month summary before any ranking happens. I verify the daily grain in query 1 below, and build the monthly aggregate explicitly in section 3.

**2. Which table(s).** `fact_content_daily_performance` (the daily fact, filtered to one month partition) for the performance signals; `dim_clients` only to sanity-check panel coverage (`gsc_data_start` / `ga4_data_start`) for the clients that show up in that month. I'm not touching `fact_content_query_90d` this week -- its 90-day window is fixed near the *end* of the snapshot (roughly April-June 2026), which overlaps or sits entirely after my March review month. Pulling a query-mix feature from it now would mean a "feature" partly built from the future, so it's excluded until I've explicitly aligned its window against whatever decision month I use.

**3. Time window.** `month=2026-03` -- a mid-panel month, not `_sample` (`_sample` is exactly June 2026, the sealed final month; the internship instructions are explicit that it's for testing query mechanics only, never for developing label logic, since the last month is the natural outcome window of any past-to-future label).

In [3]:
# Query 1 -- grain: one row really is one (report_date, client_hash_id, content_hash_id)
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM {REL_MONTH}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print(f"duplicate (date, client, content) combinations found: {len(grain_check)}")
grain_check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

duplicate (date, client, content) combinations found: 0


,report_date,client_hash_id,content_hash_id,n


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**4. What I'd predict or rank (label or proxy).** Same framing as Week 2: a ranked refresh queue, `priority_score` per content item. This week I don't build the real target yet -- that's Weeks 4-5 -- but I define it here so the leakage trap in section 3 has a real label to demonstrate on: **did this content item's impressions decline by 20%+ the month *after* my review month** (March -> April), built strictly from `fact_content_daily_performance`, never from a product flag (FlyRank's `health_score` / `priority_score` / `action_type` aren't shipped in this release at all, so there's nothing to accidentally copy -- but I still write the rule down so I don't quietly redefine it later).

**5. What I deliberately exclude.** `fact_content_query_90d` for this month (see section 1, reason 2 -- its fixed window overlaps/precedes the future relative to March) and any row from a client whose `ga4_data_start` is after the end of March -- for those rows, GA4 columns aren't "zero engagement," they're "not tracked yet," and averaging them in would quietly bias any GA4 feature toward zero for newly-onboarded clients. Query 3 below measures exactly how many March rows this affects.

**Feature / Context table for the 5 features I build in section 3:**

| Field | Bucket | Why |
|---|---|---|
| `gsc_impressions`, `gsc_clicks`, `gsc_avg_position` | Feature | Observed GSC measurements, fully inside the March window |
| `report_date`, `client_hash_id`, `content_hash_id` | Context | Grouping/joining only, never model inputs |
| `ga4_data_available` | Context (filter flag) | Used to decide which rows to keep, not as a feature itself |
| April `gsc_impressions` (any April column) | Label / proxy source | Defines the outcome; never a feature for a March-decision model |

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Query 1 (grain) is already run above and came back empty -- the grain holds. Two more facts to prove, then the five features, then the trap.

In [4]:
# Query 2 -- row count and date span for the March slice
counts = con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           COUNT(DISTINCT content_hash_id) AS n_content
    FROM {REL_MONTH}
""").df()
counts


,n_rows,min_date,max_date,n_clients,n_content
0,9841378,2026-03-01,2026-03-31,55,331437


In [5]:
# Query 3 -- availability, filtered with IS TRUE, showing how many rows survive
availability = con.sql(f"""
    SELECT COUNT(*) AS total_march_rows,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS rows_with_ga4,
           ROUND(100.0 * COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) / COUNT(*), 1) AS pct_with_ga4
    FROM {REL_MONTH}
""").df()
availability


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_march_rows,rows_with_ga4,pct_with_ga4
0,9841378,413966,4.2


**Five features (max), built only from the March window.**

1. `march_impressions` -- SUM of `gsc_impressions` across March. *Knowable at the decision moment because* every impression it sums was already logged by the time the review month ends.
2. `march_clicks` -- SUM of `gsc_clicks` across March. *Knowable at the decision moment because* same reasoning -- purely observed clicks inside the window.
3. `march_ctr` -- `march_clicks / march_impressions`, computed only from the two totals above (weighted, not an average of daily rates). *Knowable at the decision moment because* it's a ratio of two March-only totals, nothing borrowed from outside the window.
4. `march_avg_position` -- average `gsc_avg_position` across March days where a position was actually recorded (`> 0`; `0` means no data, same convention as the starter CSV). *Knowable at the decision moment because* it reflects the ranking the page already achieved during March, not a future ranking.
5. `days_with_impressions` -- count of March days with `gsc_impressions > 0`. *Knowable at the decision moment because* it's a tally of days that have already happened by month's end.

In [6]:
# five-feature frame, one row per content item, March only
features = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS march_impressions,
           SUM(gsc_clicks) AS march_clicks,
           SUM(gsc_clicks)::DOUBLE / NULLIF(SUM(gsc_impressions), 0) AS march_ctr,
           AVG(gsc_avg_position) FILTER (WHERE gsc_avg_position > 0) AS march_avg_position,
           COUNT(*) FILTER (WHERE gsc_impressions > 0) AS days_with_impressions
    FROM {REL_MONTH}
    WHERE ga4_data_available IS TRUE
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 1
""").df()

print(f"{len(features):,} content items with March visibility")
features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

63,856 content items with March visibility


,client_hash_id,content_hash_id,march_impressions,march_clicks,march_ctr,march_avg_position,days_with_impressions
0,client_65de48885f4ef01b,content_b1f61fc81b28b2d4,458.0,2.0,0.004367,4.418032,9
1,client_65de48885f4ef01b,content_e25ea7297a1dffd3,3943.0,23.0,0.005833,4.392897,25
2,client_65de48885f4ef01b,content_3c286ded8bd68120,2180.0,15.0,0.006881,8.439390,19
3,client_65de48885f4ef01b,content_b2108e8fe3360fa6,503.0,8.0,0.015905,5.531459,14
4,client_65de48885f4ef01b,content_ff867882e604fa96,24.0,0.0,0.000000,2.850000,2


## 4. The trap

*Add ONE label-derived column on purpose, watch your quick score jump toward perfect, then delete it and keep the honest number.*

**The label.** Pull April's per-content impressions the same way, and define: did this page's impressions drop 20%+ from March to April? That's a real future outcome -- it only exists because April has already happened in this snapshot; a live system wouldn't have it at decision time.

In [7]:
REL_APRIL = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/**/*.parquet')"

april = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS april_impressions
    FROM {REL_APRIL}
    GROUP BY 1, 2
""").df()

data = features.merge(april, on=["client_hash_id", "content_hash_id"], how="inner")
data = data.dropna(subset=["march_ctr", "march_avg_position"])

# require a real March base so a tiny denominator doesn't fake a "decline"
data = data[data["march_impressions"] >= 50].copy()
data["declined_next_month"] = (data["april_impressions"] < 0.8 * data["march_impressions"]).astype(int)

print(f"{len(data):,} content items with both March and April data")
print(f"base rate (share declining): {data['declined_next_month'].mean():.3f}")
data.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

38,844 content items with both March and April data
base rate (share declining): 0.049


,client_hash_id,content_hash_id,march_impressions,march_clicks,march_ctr,march_avg_position,days_with_impressions,april_impressions,declined_next_month
0,client_65de48885f4ef01b,content_b1f61fc81b28b2d4,458.0,2.0,0.004367,4.418032,9,937.0,0
1,client_65de48885f4ef01b,content_e25ea7297a1dffd3,3943.0,23.0,0.005833,4.392897,25,3616.0,0
2,client_65de48885f4ef01b,content_3c286ded8bd68120,2180.0,15.0,0.006881,8.439390,19,1437.0,1
3,client_65de48885f4ef01b,content_b2108e8fe3360fa6,503.0,8.0,0.015905,5.531459,14,376.0,1
6,client_65de48885f4ef01b,content_d6c71358297cfd6a,669.0,22.0,0.032885,3.615123,12,887.0,0


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

honest_features = ["march_impressions", "march_clicks", "march_ctr", "march_avg_position", "days_with_impressions"]

X = data[honest_features]
y = data["declined_next_month"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
honest_auc = roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])
print(f"HONEST score (March-only features) -- ROC AUC: {honest_auc:.3f}")


HONEST score (March-only features) -- ROC AUC: 0.767


In [9]:
# now spring the trap: add ONE column derived straight from the label period
data["leaked_feature"] = data["april_impressions"]  # literally the outcome month's own total

leaked_features = honest_features + ["leaked_feature"]
X_leak = data[leaked_features]
X_tr, X_te, y_tr, y_te = train_test_split(X_leak, y, test_size=0.25, random_state=42, stratify=y)

model_leak = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
leaked_auc = roc_auc_score(y_te, model_leak.predict_proba(X_te)[:, 1])
print(f"LEAKED score (April impressions smuggled in as a 'feature') -- ROC AUC: {leaked_auc:.3f}")
print(f"jump: {honest_auc:.3f} -> {leaked_auc:.3f}")


LEAKED score (April impressions smuggled in as a 'feature') -- ROC AUC: 1.000
jump: 0.767 -> 1.000


In [10]:
# delete the leaked column and keep the honest number -- this is the one that goes in the report
del data["leaked_feature"]
print(f"KEEPING the honest score: ROC AUC = {honest_auc:.3f} (March-only features, no future data)")
print("The leaked version is not a result -- it is a demonstration of why trend_direction-style")
print("current-window proxies and any next-month column must never sit in the feature set.")


KEEPING the honest score: ROC AUC = 0.767 (March-only features, no future data)
The leaked version is not a result -- it is a demonstration of why trend_direction-style
current-window proxies and any next-month column must never sit in the feature set.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation: the panel is unbalanced, and March doesn't mean the same thing for every client.** `dim_clients.gsc_data_start` differs by client -- a client whose tracking started in February 2026 has one month of March history behind it, while a client tracked since 2025 has a full year. My March slice therefore mixes clients at very different points in their own history, and a `march_ctr` or `days_with_impressions` computed the same way for both isn't really comparable across them. The `ga4_data_available IS TRUE` filter in query 3 only catches the GA4 half of this problem (rows before a client's GA4 start); it does nothing for GSC history depth. A more careful version of this contract would compute each feature relative to each client's own `gsc_data_start`, not against one shared March calendar window.

In [11]:
# how many distinct clients show up in my March slice, and how does their GSC history depth vary?
client_depth = con.sql(f"""
    SELECT c.client_hash_id, c.gsc_data_start, c.ga4_data_start
    FROM {TABLES['dim_clients']} c
    WHERE c.client_hash_id IN (SELECT DISTINCT client_hash_id FROM {REL_MONTH})
    ORDER BY c.gsc_data_start NULLS LAST
""").df()

print(f"{len(client_depth):,} distinct clients appear in the March slice")
print(f"earliest gsc_data_start: {client_depth['gsc_data_start'].min()}")
print(f"latest gsc_data_start:   {client_depth['gsc_data_start'].max()}")
client_depth.head()


55 distinct clients appear in the March slice
earliest gsc_data_start: 2025-01-27 00:00:00
latest gsc_data_start:   2026-03-27 00:00:00


,client_hash_id,gsc_data_start,ga4_data_start
0,client_9958f0a7ae1df715,2025-01-27,2025-10-29
1,client_ff644d8251367cbb,2025-01-27,2025-10-29
2,client_73cda7b4e4f265ea,2025-02-11,2026-03-24
3,client_fef1a8f436438636,2025-03-11,2026-03-06
4,client_62f4a7e64f5e0096,2025-06-07,NaT


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.